# Bronze Ingestion

- Reads the three raw files from the Unity Catalog volume `crop_risk.bronze.raw_files` and writes them into bronze Delta tables, unchanged except for audit columns. 
- No cleaning, no renaming, no deduplication happening. 
- Bronze is the replayable mirror of the source. 
- Schemas are declared explicitly rather than inferred, since schema inference silently guessed wrong types during profiling (e.g. `precipitation_hours` inferring as a decimal when it's actually a whole number in both files).

In [0]:
from pyspark.sql import functions as F

CATALOG = "crop_risk"
VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/raw_files"

1. `crop_trend_master.csv` -> `crop_risk.bronze.crop_trend_master`

In [0]:
crop_trend_schema = """
    province STRING,
    year INT,
    quarter STRING,
    temperature_2m_mean DOUBLE,
    temperature_2m_max DOUBLE,
    temperature_2m_min DOUBLE,
    precipitation_sum DOUBLE,
    rain_sum DOUBLE,
    precipitation_hours INT,
    sunshine_duration DOUBLE,
    shortwave_radiation_sum DOUBLE,
    reference_evapotranspiration_mm DOUBLE,
    wind_speed_10m_max DOUBLE,
    wind_gusts_10m_max DOUBLE,
    rain_normal DOUBLE,
    temp_normal DOUBLE,
    rainfall_deviation_pct DOUBLE,
    temperature_anomaly_c DOUBLE,
    oni_index DOUBLE,
    crop STRING,
    production DOUBLE,
    quarter_num INT,
    time_index INT,
    production_lag_1 DOUBLE,
    production_lag_2 DOUBLE,
    production_lag_4 DOUBLE,
    temp_anomaly_rolling DOUBLE,
    rain_anomaly_rolling DOUBLE,
    enso_phase_la_nina BOOLEAN,
    enso_phase_neutral BOOLEAN,
    crop_group STRING,
    growth_rate_yoy DOUBLE,
    climate_stress DOUBLE,
    risk_label STRING
"""

df_crop_trend = (
    spark.read
    .option("header", True)
    .schema(crop_trend_schema)
    .csv(f"{VOLUME_PATH}/crop_trend_master.csv")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

(
    df_crop_trend.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.bronze.crop_trend_master")
)

print(f"crop_trend_master: {df_crop_trend.count():,} rows")

2. `province_weather_quarterly.csv` -> `crop_risk.bronze.province_weather_quarterly`

In [0]:
weather_schema = """
    province STRING,
    year INT,
    year_quarter STRING,
    temperature_2m_mean DOUBLE,
    temperature_2m_max DOUBLE,
    temperature_2m_min DOUBLE,
    precipitation_sum DOUBLE,
    rain_sum DOUBLE,
    precipitation_hours INT,
    sunshine_duration DOUBLE,
    shortwave_radiation_sum DOUBLE,
    reference_evapotranspiration_mm DOUBLE,
    wind_speed_10m_max DOUBLE,
    wind_gusts_10m_max DOUBLE,
    quarter STRING,
    rain_normal DOUBLE,
    temp_normal DOUBLE,
    rainfall_deviation_pct DOUBLE,
    temperature_anomaly_c DOUBLE,
    oni_index DOUBLE,
    enso_phase STRING
"""

df_weather = (
    spark.read
    .option("header", True)
    .schema(weather_schema)
    .csv(f"{VOLUME_PATH}/province_weather_quarterly.csv")
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

(
    df_weather.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.bronze.province_weather_quarterly")
)

print(f"province_weather_quarterly: {df_weather.count():,} rows")

3. `data_dictionary.csv` -> `crop_risk.bronze.data_dictionary`

In [0]:
dictionary_schema = """
    file STRING,
    column_name STRING,
    data_type STRING,
    unit STRING,
    derived STRING,
    description STRING
"""

df_dictionary = (
    spark.read
    .option("header", True)
    .schema(dictionary_schema)
    .csv(f"{VOLUME_PATH}/data_dictionary.csv")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

(
    df_dictionary.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.bronze.data_dictionary")
)
    
print(f"data_dictionary: {df_dictionary.count():,} rows")

4. sanity check

Confirm row counts match the source files before moving to silver. Expected: crop_trend_master 167,699 rows, province_weather_quarterly: 3,520 rows, data_dictionary: 55 rows.

In [0]:
for table in ["crop_trend_master", "province_weather_quarterly", "data_dictionary"]:
    n = spark.table(f"{CATALOG}.bronze.{table}").count()
    print(f"{CATALOG}.bronze.{table}: {n:,} rows")